In [1]:
# Gerekli kütüphaneler (Colab'da ilk kez çalıştırırken gerekirse açın)
# !pip install -q sgp4 pandas matplotlib ipywidgets

import io
import math
import urllib.request
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
from sgp4.api import Satrec

MU_EARTH = 398600.4418  # km^3 / s^2
R_EARTH = 6378.137  # km

SOURCES = {
    "Active Satellites (İlk 300)": (
        "https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle",
        300,
    ),
    "Space Stations": (
        "https://celestrak.org/NORAD/elements/gp.php?GROUP=stations&FORMAT=tle",
        50,
    ),
    "Starlink (İlk 300)": (
        "https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle",
        300,
    ),
}


def parse_tle_text(tle_text, max_sats=300):
  lines = [ln.strip() for ln in tle_text.strip().splitlines() if ln.strip()]
  rows = []
  i = 0
  while i + 1 < len(lines) and len(rows) < max_sats:
    if (
        not lines[i].startswith("1 ")
        and lines[i + 1].startswith("1 ")
        and (i + 2) < len(lines)
        and lines[i + 2].startswith("2 ")
    ):
      name, l1, l2 = lines[i], lines[i + 1], lines[i + 2]
      i += 3
    elif lines[i].startswith("1 ") and (i + 1) < len(lines) and lines[i + 1].startswith("2 "):
      l1, l2 = lines[i], lines[i + 1]
      name = f"SAT-{l1[2:7].strip()}"
      i += 2
    else:
      i += 1
      continue

    sat = Satrec.twoline2rv(l1, l2)
    inc_deg = math.degrees(sat.inclo)
    ecc = sat.ecco
    # sat.no_kozai rad/min cinsindendir
    n_rad_s = sat.no_kozai / 60.0
    if n_rad_s <= 0:
      continue
    period_min = (2 * math.pi / n_rad_s) / 60.0
    a_km = (MU_EARTH / (n_rad_s**2)) ** (1 / 3)
    perigee_alt = a_km * (1 - ecc) - R_EARTH
    apogee_alt = a_km * (1 + ecc) - R_EARTH

    rows.append({
        "name": name,
        "norad_id": sat.satnum,
        "inclination_deg": round(inc_deg, 3),
        "eccentricity": round(ecc, 5),
        "period_min": round(period_min, 2),
        "perigee_alt_km": round(perigee_alt, 1),
        "apogee_alt_km": round(apogee_alt, 1),
        "mean_alt_km": round((perigee_alt + apogee_alt) / 2, 1),
        "bstar": sat.bstar,
    })
  return pd.DataFrame(rows)


def generate_commentary(df, row):
  notes = []
  inc = row["inclination_deg"]
  ecc = row["eccentricity"]
  alt = row["mean_alt_km"]
  bstar = row["bstar"]

  # 1. Yörünge rejimine göre yorum
  if alt < 2000:
    regime = "Alçak Dünya Yörüngesi (LEO)"
  elif alt < 35000:
    regime = "Orta Dünya Yörüngesi (MEO)"
  else:
    regime = "Jeostatik / Yüksek Yörünge (GEO/HEO)"

  notes.append(
      f"- **Rejim ve İrtifa:** {row['name']} yaklaşık **{alt} km** ortalama"
      f" irtifada ({regime}) ve **{row['period_min']} dk** periyotla"
      " dolanıyor."
  )

  # 2. Eğim örüntüsü
  if 96 <= inc <= 102 and alt < 1200:
    notes.append(
        f"- **Örüntü İpucu (Güneş Eşzamanlı Bant):** Eğim açısı **{inc}°** ve"
        " irtifası LEO bandında. Bu dar aralık, düğüm çizgisinin (RAAN) yıl"
        " boyunca Dünya'nın Güneş etrafındaki dönüş hızıyla (~0.9856°/gün)"
        " kaymasını sağlayarak aydınlatma koşullarını sabit tutar."
    )
  elif 50 <= inc <= 55 and 300 <= alt <= 600:
    notes.append(
        f"- **Örüntü İpucu (Takımyıldız / Fırlatma Koridoru):** **{inc}°**"
        " civarı eğim, ticari takımyıldızlar veya belirli fırlatma"
        " eğimleriyle uyumlu yoğun bir küme içinde yer alıyor."
    )
  elif inc > 90:
    notes.append(
        f"- **Yön (Retrograd):** Eğim **{inc}°** olduğu için uydu Dünya'nın"
        " dönüş yönünün tersine hareket ediyor."
    )

  # 3. Dışmerkezlik ve sürüklenme
  if ecc > 0.05:
    notes.append(
        f"- **Eliptiklik:** Dışmerkezliği **{ecc}** ile dairesel yörüngeden"
        f" belirgin biçimde sapıyor (yerberi: {row['perigee_alt_km']} km,"
        f" yerötesi: {row['apogee_alt_km']} km)."
    )
  else:
    notes.append(
        f"- **Dairesellik:** Dışmerkezliği **{ecc}**; neredeyse dairesel bir"
        " yörünge izliyor."
    )

  notes.append(
      f"- **Atmosferik Sürüklenme (`BSTAR`):** Değeri `{bstar:.2e}`."
      " Alçak irtifada pozitif ve yüksek büyüklükteki değerler, artık"
      " atmosfer yoğunluğunun yörünge ömrü üzerindeki etkisini gösterir."
  )
  return "\n".join(notes)


# --- Arayüz Elemanları ---
source_dd = widgets.Dropdown(
    options=list(SOURCES.keys()), description="Grup:"
)
load_btn = widgets.Button(description="Veriyi Çek", button_style="primary")
sat_dd = widgets.Dropdown(options=[], description="Uydu Seç:")
analyze_btn = widgets.Button(
    description="Analiz Et & Yorumla", button_style="success"
)
custom_tle_ta = widgets.Textarea(
    placeholder="İstersen buraya doğrudan 2 veya 3 satırlı TLE verisi yapıştır...",
    layout=widgets.Layout(width="100%", height="90px"),
)
use_custom_btn = widgets.Button(description="Yapıştırılan TLE'yi Kullan")
out = widgets.Output()

state = {"df": pd.DataFrame()}


def populate_sats(df):
  state["df"] = df
  sat_dd.options = [
      (f"{r['name']} ({r['norad_id']})", idx) for idx, r in df.iterrows()
  ]


def on_load(_):
  with out:
    clear_output()
    url, limit = SOURCES[source_dd.value]
    try:
      req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
      raw = urllib.request.urlopen(req, timeout=15).read().decode("utf-8")
      df = parse_tle_text(raw, max_sats=limit)
      populate_sats(df)
      print(f"Başarılı: {len(df)} uydu yüklendi. Şimdi bir uydu seçip analiz et.")
    except Exception as e:
      print("Veri çekilirken hata oluştu:", e)


def on_custom(_):
  with out:
    clear_output()
    df = parse_tle_text(custom_tle_ta.value, max_sats=100)
    if df.empty:
      print("Geçerli TLE satırı bulunamadı. Satır 1 ve Satır 2 formatını kontrol et.")
    else:
      populate_sats(df)
      print(f"{len(df)} adet özel TLE kaydı yüklendi.")


def on_analyze(_):
  df = state["df"]
  if df.empty or sat_dd.value is None:
    return
  row = df.loc[sat_dd.value]
  with out:
    clear_output()
    print("=== SEÇİLİ UYDU ÖZETİ ===")
    display(pd.DataFrame([row]))
    print("\n=== OTOMATİK YORUM VE ÖRÜNTÜ ANALİZİ ===")
    print(generate_commentary(df, row))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.scatter(
        df["inclination_deg"],
        df["mean_alt_km"],
        alpha=0.45,
        s=18,
        label="Gruptaki Diğer Uydular",
    )
    ax.scatter(
        [row["inclination_deg"]],
        [row["mean_alt_km"]],
        color="red",
        s=90,
        zorder=5,
        label=f"Seçili: {row['name']}",
    )
    ax.set_xlabel("Eğim Açısı (derece)")
    ax.set_ylabel("Ortalama İrtifa (km)")
    ax.set_title("Grup İçinde Eğim - İrtifa Dağılımı")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()
    plt.tight_layout()
    plt.show()


load_btn.on_click(on_load)
use_custom_btn.on_click(on_custom)
analyze_btn.on_click(on_analyze)

ui = widgets.VBox([
    widgets.HBox([source_dd, load_btn]),
    widgets.HBox([sat_dd, analyze_btn]),
    custom_tle_ta,
    use_custom_btn,
    out,
])
display(ui)

ModuleNotFoundError: No module named 'sgp4'

In [2]:
!pip install -q sgp4

In [3]:
# Gerekli kütüphaneler (Colab'da ilk kez çalıştırırken gerekirse açın)
# !pip install -q sgp4 pandas matplotlib ipywidgets

import io
import math
import urllib.request
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
from sgp4.api import Satrec

MU_EARTH = 398600.4418  # km^3 / s^2
R_EARTH = 6378.137  # km

SOURCES = {
    "Active Satellites (İlk 300)": (
        "https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle",
        300,
    ),
    "Space Stations": (
        "https://celestrak.org/NORAD/elements/gp.php?GROUP=stations&FORMAT=tle",
        50,
    ),
    "Starlink (İlk 300)": (
        "https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle",
        300,
    ),
}


def parse_tle_text(tle_text, max_sats=300):
  lines = [ln.strip() for ln in tle_text.strip().splitlines() if ln.strip()]
  rows = []
  i = 0
  while i + 1 < len(lines) and len(rows) < max_sats:
    if (
        not lines[i].startswith("1 ")
        and lines[i + 1].startswith("1 ")
        and (i + 2) < len(lines)
        and lines[i + 2].startswith("2 ")
    ):
      name, l1, l2 = lines[i], lines[i + 1], lines[i + 2]
      i += 3
    elif lines[i].startswith("1 ") and (i + 1) < len(lines) and lines[i + 1].startswith("2 "):
      l1, l2 = lines[i], lines[i + 1]
      name = f"SAT-{l1[2:7].strip()}"
      i += 2
    else:
      i += 1
      continue

    sat = Satrec.twoline2rv(l1, l2)
    inc_deg = math.degrees(sat.inclo)
    ecc = sat.ecco
    # sat.no_kozai rad/min cinsindendir
    n_rad_s = sat.no_kozai / 60.0
    if n_rad_s <= 0:
      continue
    period_min = (2 * math.pi / n_rad_s) / 60.0
    a_km = (MU_EARTH / (n_rad_s**2)) ** (1 / 3)
    perigee_alt = a_km * (1 - ecc) - R_EARTH
    apogee_alt = a_km * (1 + ecc) - R_EARTH

    rows.append({
        "name": name,
        "norad_id": sat.satnum,
        "inclination_deg": round(inc_deg, 3),
        "eccentricity": round(ecc, 5),
        "period_min": round(period_min, 2),
        "perigee_alt_km": round(perigee_alt, 1),
        "apogee_alt_km": round(apogee_alt, 1),
        "mean_alt_km": round((perigee_alt + apogee_alt) / 2, 1),
        "bstar": sat.bstar,
    })
  return pd.DataFrame(rows)


def generate_commentary(df, row):
  notes = []
  inc = row["inclination_deg"]
  ecc = row["eccentricity"]
  alt = row["mean_alt_km"]
  bstar = row["bstar"]

  # 1. Yörünge rejimine göre yorum
  if alt < 2000:
    regime = "Alçak Dünya Yörüngesi (LEO)"
  elif alt < 35000:
    regime = "Orta Dünya Yörüngesi (MEO)"
  else:
    regime = "Jeostatik / Yüksek Yörünge (GEO/HEO)"

  notes.append(
      f"- **Rejim ve İrtifa:** {row['name']} yaklaşık **{alt} km** ortalama"
      f" irtifada ({regime}) ve **{row['period_min']} dk** periyotla"
      " dolanıyor."
  )

  # 2. Eğim örüntüsü
  if 96 <= inc <= 102 and alt < 1200:
    notes.append(
        f"- **Örüntü İpucu (Güneş Eşzamanlı Bant):** Eğim açısı **{inc}°** ve"
        " irtifası LEO bandında. Bu dar aralık, düğüm çizgisinin (RAAN) yıl"
        " boyunca Dünya'nın Güneş etrafındaki dönüş hızıyla (~0.9856°/gün)"
        " kaymasını sağlayarak aydınlatma koşullarını sabit tutar."
    )
  elif 50 <= inc <= 55 and 300 <= alt <= 600:
    notes.append(
        f"- **Örüntü İpucu (Takımyıldız / Fırlatma Koridoru):** **{inc}°**"
        " civarı eğim, ticari takımyıldızlar veya belirli fırlatma"
        " eğimleriyle uyumlu yoğun bir küme içinde yer alıyor."
    )
  elif inc > 90:
    notes.append(
        f"- **Yön (Retrograd):** Eğim **{inc}°** olduğu için uydu Dünya'nın"
        " dönüş yönünün tersine hareket ediyor."
    )

  # 3. Dışmerkezlik ve sürüklenme
  if ecc > 0.05:
    notes.append(
        f"- **Eliptiklik:** Dışmerkezliği **{ecc}** ile dairesel yörüngeden"
        f" belirgin biçimde sapıyor (yerberi: {row['perigee_alt_km']} km,"
        f" yerötesi: {row['apogee_alt_km']} km)."
    )
  else:
    notes.append(
        f"- **Dairesellik:** Dışmerkezliği **{ecc}**; neredeyse dairesel bir"
        " yörünge izliyor."
    )

  notes.append(
      f"- **Atmosferik Sürüklenme (`BSTAR`):** Değeri `{bstar:.2e}`."
      " Alçak irtifada pozitif ve yüksek büyüklükteki değerler, artık"
      " atmosfer yoğunluğunun yörünge ömrü üzerindeki etkisini gösterir."
  )
  return "\n".join(notes)


# --- Arayüz Elemanları ---
source_dd = widgets.Dropdown(
    options=list(SOURCES.keys()), description="Grup:"
)
load_btn = widgets.Button(description="Veriyi Çek", button_style="primary")
sat_dd = widgets.Dropdown(options=[], description="Uydu Seç:")
analyze_btn = widgets.Button(
    description="Analiz Et & Yorumla", button_style="success"
)
custom_tle_ta = widgets.Textarea(
    placeholder="İstersen buraya doğrudan 2 veya 3 satırlı TLE verisi yapıştır...",
    layout=widgets.Layout(width="100%", height="90px"),
)
use_custom_btn = widgets.Button(description="Yapıştırılan TLE'yi Kullan")
out = widgets.Output()

state = {"df": pd.DataFrame()}


def populate_sats(df):
  state["df"] = df
  sat_dd.options = [
      (f"{r['name']} ({r['norad_id']})", idx) for idx, r in df.iterrows()
  ]


def on_load(_):
  with out:
    clear_output()
    url, limit = SOURCES[source_dd.value]
    try:
      req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
      raw = urllib.request.urlopen(req, timeout=15).read().decode("utf-8")
      df = parse_tle_text(raw, max_sats=limit)
      populate_sats(df)
      print(f"Başarılı: {len(df)} uydu yüklendi. Şimdi bir uydu seçip analiz et.")
    except Exception as e:
      print("Veri çekilirken hata oluştu:", e)


def on_custom(_):
  with out:
    clear_output()
    df = parse_tle_text(custom_tle_ta.value, max_sats=100)
    if df.empty:
      print("Geçerli TLE satırı bulunamadı. Satır 1 ve Satır 2 formatını kontrol et.")
    else:
      populate_sats(df)
      print(f"{len(df)} adet özel TLE kaydı yüklendi.")


def on_analyze(_):
  df = state["df"]
  if df.empty or sat_dd.value is None:
    return
  row = df.loc[sat_dd.value]
  with out:
    clear_output()
    print("=== SEÇİLİ UYDU ÖZETİ ===")
    display(pd.DataFrame([row]))
    print("\n=== OTOMATİK YORUM VE ÖRÜNTÜ ANALİZİ ===")
    print(generate_commentary(df, row))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.scatter(
        df["inclination_deg"],
        df["mean_alt_km"],
        alpha=0.45,
        s=18,
        label="Gruptaki Diğer Uydular",
    )
    ax.scatter(
        [row["inclination_deg"]],
        [row["mean_alt_km"]],
        color="red",
        s=90,
        zorder=5,
        label=f"Seçili: {row['name']}",
    )
    ax.set_xlabel("Eğim Açısı (derece)")
    ax.set_ylabel("Ortalama İrtifa (km)")
    ax.set_title("Grup İçinde Eğim - İrtifa Dağılımı")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()
    plt.tight_layout()
    plt.show()


load_btn.on_click(on_load)
use_custom_btn.on_click(on_custom)
analyze_btn.on_click(on_analyze)

ui = widgets.VBox([
    widgets.HBox([source_dd, load_btn]),
    widgets.HBox([sat_dd, analyze_btn]),
    custom_tle_ta,
    use_custom_btn,
    out,
])
display(ui)

In [4]:
!pip install -q sgp4 gradio pandas

import math
import gradio as gr
import pandas as pd
from sgp4.api import Satrec

MU_EARTH = 398600.4418
R_EARTH = 6378.137

CATEGORIES = {
    "Special-Interest Satellites → 100 (or so) Brightest": (
        "100 (or so) Brightest"
    ),
    "Special-Interest Satellites → Space Stations": "Space Stations",
    "Special-Interest Satellites → Active Satellites": "Active Satellites",
    "Weather & Earth Resources Satellites → Weather": "Weather",
    "Weather & Earth Resources Satellites → Earth Resources": (
        "Earth Resources"
    ),
    "Communications Satellites → Starlink": "Starlink",
    "Navigation Satellites → GPS Operational": "GPS Operational",
    "Miscellaneous Satellites → CubeSats": "CubeSats",
}

UI_TEXT = {
    "TR": {
        "title": "Uydu Yörünge & Örüntü Analiz Paneli",
        "subtitle": (
            "CelesTrak TLE Verisi • İki Dilli Analiz (TR/EN) • Örüntü Keşfi"
        ),
        "lang": "Dil / Language",
        "cat": "CelesTrak Başlık / Kategori Seçimi",
        "tle_box": "TLE Verisini Buraya Yapıştır (2 veya 3 Satırlı)",
        "tle_ph": (
            "CelesTrak'tan kopyaladığın TLE satırlarını buraya yapıştır..."
        ),
        "sat_dd": "Gruptaki Uyduyu Seç",
        "btn_run": "⚡ Analiz Et & Örüntüyü Yorumla",
        "no_data": (
            "⚠️ Geçerli TLE satırı bulunamadı. Satır 1 ve Satır 2 formatını"
            " kontrol et."
        ),
        "regime_leo": "Alçak Dünya Yörüngesi (LEO)",
        "regime_meo": "Orta Dünya Yörüngesi (MEO)",
        "regime_geo": "Yüksek / Jeostatik Rejim (GEO/HEO)",
        "pattern_sso": (
            "Güneş Eşzamanlı Bant (SSO): Eğim {inc:.2f}° ve irtifa"
            " {alt:.0f} km civarında. Düğüm çizgisinin (RAAN) yıl boyunca"
            " kayması, uydunun Dünya yüzeyinden her geçişte benzer güneş ışığı"
            " açısıyla geçmesini sağlar."
        ),
        "pattern_const": (
            "Takımyıldız / Fırlatma Koridoru: {inc:.2f}° eğim ve {alt:.0f} km"
            " irtifa bandında yoğunlaşma görülüyor; bu yapı çoklu uydu"
            " kapsama veya ortak fırlatma eğimi mantığını yansıtır."
        ),
        "pattern_ecc": (
            "Yüksek Dışmerkezlik / Eliptik Yörünge: e = {ecc:.4f} (Yerberi:"
            " {per:.0f} km, Yerötesi: {apo:.0f} km). Dairesel yörüngelerden"
            " ayrılan bu yapı, belirli geçitlerde bekleme veya transfer/enkaz"
            " dinamiğine işaret eder."
        ),
        "pattern_group": (
            "Grup Dağılım Örüntüsü: İncelenen grupta ortalama eğim"
            " {m_inc:.2f}° ve ortalama irtifa {m_alt:.0f} km. Seçili uydu"
            " grubun ortalama irtifasından {diff:+.0f} km farklı konumda."
        ),
        "drag_note": (
            "Atmosfer Sürüklenmesi (BSTAR): {bstar:.2e}. Alçak irtifada"
            " pozitif/büyük değerler üst atmosfer yoğunluğunun yörünge"
            " alçalmasına etkisini gösterir."
        ),
    },
    "EN": {
        "title": "Satellite Orbit & Pattern Analysis Studio",
        "subtitle": (
            "CelesTrak TLE Data • Bilingual Analysis (TR/EN) • Pattern"
            " Discovery"
        ),
        "lang": "Dil / Language",
        "cat": "CelesTrak Category / Header Selection",
        "tle_box": "Paste TLE Data Here (2-Line or 3-Line)",
        "tle_ph": "Paste TLE lines copied from CelesTrak here...",
        "sat_dd": "Select Satellite in Group",
        "btn_run": "⚡ Run Orbit & Pattern Analysis",
        "no_data": (
            "⚠️ No valid TLE lines found. Please check Line 1 and Line 2"
            " formatting."
        ),
        "regime_leo": "Low Earth Orbit (LEO)",
        "regime_meo": "Medium Earth Orbit (MEO)",
        "regime_geo": "High / Geostationary Regime (GEO/HEO)",
        "pattern_sso": (
            "Sun-Synchronous Band (SSO): Inclination {inc:.2f}° near"
            " {alt:.0f} km altitude. Nodal precession (RAAN drift) keeps"
            " lighting geometry consistent across passes."
        ),
        "pattern_const": (
            "Constellation / Launch Corridor: Clustered around {inc:.2f}°"
            " inclination and {alt:.0f} km altitude, reflecting coordinated"
            " coverage or shared launch inclination."
        ),
        "pattern_ecc": (
            "Elevated Eccentricity / Elliptical Orbit: e = {ecc:.4f} (Perigee:"
            " {per:.0f} km, Apogee: {apo:.0f} km). Stands out from circular"
            " orbits due to dwell geometry, transfer, or debris dynamics."
        ),
        "pattern_group": (
            "Group Distribution Pattern: Mean inclination is {m_inc:.2f}° and"
            " mean altitude is {m_alt:.0f} km across this set; selected"
            " satellite sits {diff:+.0f} km relative to group mean."
        ),
        "drag_note": (
            "Atmospheric Drag (BSTAR): {bstar:.2e}. Higher positive values at"
            " lower altitudes indicate stronger upper-atmosphere drag effects."
        ),
    },
}


def parse_tle(text):
  lines = [ln.strip() for ln in (text or "").strip().splitlines() if ln.strip()]
  rows = []
  i = 0
  while i + 1 < len(lines):
    if (
        not lines[i].startswith("1 ")
        and lines[i + 1].startswith("1 ")
        and (i + 2) < len(lines)
        and lines[i + 2].startswith("2 ")
    ):
      name, l1, l2 = lines[i], lines[i + 1], lines[i + 2]
      i += 3
    elif lines[i].startswith("1 ") and (i + 1) < len(lines) and lines[i + 1].startswith("2 "):
      l1, l2 = lines[i], lines[i + 1]
      name = f"SAT-{l1[2:7].strip()}"
      i += 2
    else:
      i += 1
      continue
    try:
      sat = Satrec.twoline2rv(l1, l2)
      n_rad_s = sat.no_kozai / 60.0
      if n_rad_s <= 0:
        continue
      inc = math.degrees(sat.inclo)
      ecc = sat.ecco
      period = (2 * math.pi / n_rad_s) / 60.0
      a_km = (MU_EARTH / (n_rad_s**2)) ** (1 / 3)
      per = a_km * (1 - ecc) - R_EARTH
      apo = a_km * (1 + ecc) - R_EARTH
      alt = (per + apo) / 2
      rows.append({
          "name": name,
          "norad_id": sat.satnum,
          "inc_deg": inc,
          "ecc": ecc,
          "period_min": period,
          "perigee_km": per,
          "apogee_km": apo,
          "alt_km": alt,
          "bstar": sat.bstar,
      })
    except Exception:
      continue
  return pd.DataFrame(rows)


def sync_sat_dropdown(tle_text):
  df = parse_tle(tle_text)
  if df.empty:
    return gr.update(choices=[], value=None)
  choices = [
      f"{r['name']} ({r['norad_id']})" for _, r in df.head(150).iterrows()
  ]
  return gr.update(choices=choices, value=choices[0])


def run_analysis(lang, category_label, tle_text, selected_sat):
  t = UI_TEXT[lang]
  df = parse_tle(tle_text)
  if df.empty:
    empty_html = (
        f"<div style='padding:16px;border-radius:10px;background:#161b26;color:#f87171;border:1px solid #334155;'>{t['no_data']}</div>"
    )
    return empty_html, empty_html

  idx = 0
  if selected_sat:
    matches = df.index[
        df.apply(lambda r: f"{r['name']} ({r['norad_id']})" == selected_sat, axis=1)
    ].tolist()
    if matches:
      idx = matches[0]
  row = df.loc[idx]

  if row["alt_km"] < 2000:
    regime = t["regime_leo"]
  elif row["alt_km"] < 35000:
    regime = t["regime_meo"]
  else:
    regime = t["regime_geo"]

  # Örüntü seçimi
  if row["ecc"] > 0.05:
    pattern_desc = t["pattern_ecc"].format(
        ecc=row["ecc"], per=row["perigee_km"], apo=row["apogee_km"]
    )
    pattern_tag = (
        "ELLIPTICAL / ECCENTRIC" if lang == "EN" else "ELİPTİK YÖRÜNGE"
    )
  elif 96 <= row["inc_deg"] <= 102 and row["alt_km"] < 1200:
    pattern_desc = t["pattern_sso"].format(
        inc=row["inc_deg"], alt=row["alt_km"]
    )
    pattern_tag = "SUN-SYNCHRONOUS BAND" if lang == "EN" else "SSO BANDI"
  elif 45 <= row["inc_deg"] <= 56 and 250 <= row["alt_km"] <= 1300:
    pattern_desc = t["pattern_const"].format(
        inc=row["inc_deg"], alt=row["alt_km"]
    )
    pattern_tag = (
        "CONSTELLATION / CORRIDOR" if lang == "EN" else "TAKIYILDIZ / BAND"
    )
  else:
    pattern_desc = t["pattern_group"].format(
        m_inc=df["inc_deg"].mean(),
        m_alt=df["alt_km"].mean(),
        diff=row["alt_km"] - df["alt_km"].mean(),
    )
    pattern_tag = "GROUP PATTERN" if lang == "EN" else "GRUP ÖRÜNTÜSÜ"

  drag_desc = t["drag_note"].format(bstar=row["bstar"])

  cards_html = f"""
    <div style="font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:12px;color:#38bdf8;font-weight:600;letter-spacing:0.4px;">{category_label}</div>
          <div style="font-size:20px;font-weight:700;color:#f8fafc;margin-top:2px;">{row['name']} <span style="font-size:13px;color:#94a3b8;">(NORAD {row['norad_id']})</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:5px 12px;border-radius:999px;font-size:12px;font-weight:700;">{regime}</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">MEAN ALTITUDE / İRTİFA</div>
          <div style="font-size:18px;font-weight:700;color:#38bdf8;margin-top:4px;">{row['alt_km']:.1f} km</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">INCLINATION / EĞİM</div>
          <div style="font-size:18px;font-weight:700;color:#a78bfa;margin-top:4px;">{row['inc_deg']:.2f}°</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ECCENTRICITY / DIŞMERKEZLİK</div>
          <div style="font-size:18px;font-weight:700;color:#34d399;margin-top:4px;">{row['ecc']:.5f}</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">PERIOD / PERİYOT</div>
          <div style="font-size:18px;font-weight:700;color:#fbbf24;margin-top:4px;">{row['period_min']:.1f} min</div>
        </div>
      </div>
    </div>
    """

  interp_html = f"""
    <div style="font-family:Inter,system-ui,sans-serif;background:#111724;border:1px solid #263248;border-radius:12px;padding:16px 18px;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;">
        <span style="font-size:14px;font-weight:700;color:#f8fafc;">🛰️ {"PATTERN & ASTRODYNAMICS INTERPRETATION" if lang=="EN" else "ÖRÜNTÜ VE ASTRODİNAMİK YORUM"}</span>
        <span style="background:#1e1b4b;color:#c4b5fd;border:1px solid #6366f1;padding:3px 10px;border-radius:999px;font-size:11px;font-weight:700;">{pattern_tag}</span>
      </div>
      <p style="margin:0 0 10px 0;line-height:1.55;color:#cbd5e1;font-size:14px;">{pattern_desc}</p>
      <p style="margin:0;line-height:1.5;color:#94a3b8;font-size:13px;">{drag_desc}</p>
    </div>
    """
  return cards_html, interp_html


custom_css = """
body, .gradio-container { background-color: #0b0f17 !important; color: #e2e8f0 !important; }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
  with gr.Row():
    with gr.Column(scale=4):
      gr.Markdown("## 🛰️ Uydu Yörünge & Örüntü Analizi / Orbit Pattern Studio")
    with gr.Column(scale=1):
      lang_dd = gr.Dropdown(
          choices=["TR", "EN"], value="TR", label="Dil / Language"
      )

  with gr.Row():
    with gr.Column(scale=1):
      cat_dd = gr.Dropdown(
          choices=list(CATEGORIES.keys()),
          value="Special-Interest Satellites → 100 (or so) Brightest",
          label="CelesTrak Başlık / Kategori",
      )
      tle_in = gr.Textbox(
          lines=9,
          label="TLE Verisini Yapıştır / Paste TLE",
          placeholder=(
              "CelesTrak'tan kopyaladığın TLE verisini buraya yapıştır..."
          ),
      )
      sat_dd = gr.Dropdown(
          choices=[], value=None, label="Uydu Seç / Select Satellite"
      )
      run_btn = gr.Button(
          "⚡ Analiz Et & Yorumla / Analyze", variant="primary"
      )

    with gr.Column(scale=2):
      metrics_out = gr.HTML()
      commentary_out = gr.HTML()

  tle_in.change(fn=sync_sat_dropdown, inputs=[tle_in], outputs=[sat_dd])
  run_btn.click(
      fn=run_analysis,
      inputs=[lang_dd, cat_dd, tle_in, sat_dd],
      outputs=[metrics_out, commentary_out],
  )

demo.launch(share=False)

/tmp/ipykernel_2830/2382229133.py:283: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [5]:
!pip install -q sgp4 gradio pandas

import math
import gradio as gr
import pandas as pd
from sgp4.api import Satrec

MU_EARTH = 398600.4418
R_EARTH = 6378.137

# Kullanıcı liste yapıştırmasa bile kategoriye göre hızlı başlangıç sağlayacak örnek TLE blokları
SAMPLE_TLE_BY_CATEGORY = {
    "Special-Interest Satellites → 100 (or so) Brightest": """ISS (ZARYA)
1 25544U 98067A   26328.54321012  .00016717  00000-0  30124-3 0  9993
2 25544  51.6392 142.8741 0007412  84.3120 275.8410 15.49812345482104
TIANGONG
1 48274U 21035A   26328.49182736  .00018210  00000-0  21450-3 0  9998
2 48274  41.4721 218.6512 0005120 112.4410 247.7102 15.61204912310459
HUBBLE SPACE TELESCOPE
1 20580U 90037B   26328.51204918  .00004120  00000-0  19842-3 0  9991
2 20580  28.4690  95.1204 0002814 210.8812 149.1801 15.09128471948212""",
    "Special-Interest Satellites → Space Stations": """ISS (ZARYA)
1 25544U 98067A   26328.54321012  .00016717  00000-0  30124-3 0  9993
2 25544  51.6392 142.8741 0007412  84.3120 275.8410 15.49812345482104
TIANGONG
1 48274U 21035A   26328.49182736  .00018210  00000-0  21450-3 0  9998
2 48274  41.4721 218.6512 0005120 112.4410 247.7102 15.61204912310459""",
    "Weather & Earth Resources Satellites → Weather": """NOAA 20 (JPSS-1)
1 43013U 17073A   26328.61029384  .00000112  00000-0  64210-4 0  9995
2 43013  98.7412  64.2190 0001420  98.4120 261.7180 14.19561284461209
METOP-C
1 43689U 18087A   26328.58910234  .00000095  00000-0  58120-4 0  9992
2 43689  98.7021 118.9042 0001215 104.2210 255.9012 14.21409812421983""",
    "Weather & Earth Resources Satellites → Earth Resources": """LANDSAT 9
1 49260U 21088A   26328.52341098  .00000184  00000-0  91045-4 0  9994
2 49260  98.2214  41.7820 0001380  89.6412 270.4910 14.57120491278415""",
    "Communications Satellites → Starlink": """STARLINK-30124
1 57123U 23084A   26328.67812345  .00001420  00000-0  11204-3 0  9996
2 57123  53.1604 184.3120 0001480  78.9120 281.2104 15.06319284192837""",
    "Navigation Satellites → GPS Operational": """GPS BIIR-11 (SVN 59)
1 28190U 04009A   26328.41209834 -.00000054  00000-0  00000+0 0  9997
2 28190  55.1204 294.1820 0084120 214.9012 144.8120  2.00564128162948""",
}


def parse_tle(text):
  lines = [ln.strip() for ln in (text or "").strip().splitlines() if ln.strip()]
  rows = []
  i = 0
  while i + 1 < len(lines):
    if (
        not lines[i].startswith("1 ")
        and lines[i + 1].startswith("1 ")
        and (i + 2) < len(lines)
        and lines[i + 2].startswith("2 ")
    ):
      name, l1, l2 = lines[i], lines[i + 1], lines[i + 2]
      i += 3
    elif lines[i].startswith("1 ") and (i + 1) < len(lines) and lines[i + 1].startswith("2 "):
      l1, l2 = lines[i], lines[i + 1]
      name = f"SAT-{l1[2:7].strip()}"
      i += 2
    else:
      i += 1
      continue
    try:
      sat = Satrec.twoline2rv(l1, l2)
      n_rad_s = sat.no_kozai / 60.0
      if n_rad_s <= 0:
        continue
      inc = math.degrees(sat.inclo)
      ecc = sat.ecco
      period = (2 * math.pi / n_rad_s) / 60.0
      a_km = (MU_EARTH / (n_rad_s**2)) ** (1 / 3)
      per = a_km * (1 - ecc) - R_EARTH
      apo = a_km * (1 + ecc) - R_EARTH
      alt = (per + apo) / 2
      rows.append({
          "name": name,
          "norad_id": sat.satnum,
          "inc_deg": inc,
          "ecc": ecc,
          "period_min": period,
          "perigee_km": per,
          "apogee_km": apo,
          "alt_km": alt,
          "bstar": sat.bstar,
      })
    except Exception:
      continue
  return pd.DataFrame(rows)


def on_category_change(category_label):
  sample = SAMPLE_TLE_BY_CATEGORY.get(category_label, "")
  df = parse_tle(sample)
  choices = [f"{r['name']} ({r['norad_id']})" for _, r in df.iterrows()]
  val = choices[0] if choices else None
  return sample, gr.update(choices=choices, value=val)


def sync_sat_dropdown(tle_text):
  df = parse_tle(tle_text)
  if df.empty:
    return gr.update(choices=[], value=None)
  choices = [
      f"{r['name']} ({r['norad_id']})" for _, r in df.head(200).iterrows()
  ]
  return gr.update(choices=choices, value=choices[0])


def run_analysis(category_label, tle_text, selected_sat):
  effective_tle = (
      tle_text.strip()
      if (tle_text and tle_text.strip())
      else SAMPLE_TLE_BY_CATEGORY.get(category_label, "")
  )
  df = parse_tle(effective_tle)
  if df.empty:
    err_html = """
        <div style="padding:16px;border-radius:10px;background:#161b26;color:#f87171;border:1px solid #334155;">
          No valid TLE lines found. Paste valid Line 1 and Line 2 entries from CelesTrak.
        </div>
        """
    return err_html, err_html

  idx = 0
  if selected_sat:
    matches = df.index[
        df.apply(lambda r: f"{r['name']} ({r['norad_id']})" == selected_sat, axis=1)
    ].tolist()
    if matches:
      idx = matches[0]
  row = df.loc[idx]

  if row["alt_km"] < 2000:
    regime = "Low Earth Orbit (LEO)"
  elif row["alt_km"] < 35000:
    regime = "Medium Earth Orbit (MEO)"
  else:
    regime = "High / Geostationary Regime (GEO/HEO)"

  if row["ecc"] > 0.05:
    pattern_tag = "SURPRISING PATTERN: ECCENTRIC ORBIT"
    pattern_desc = (
        f"With an eccentricity of <b>{row['ecc']:.4f}</b> (perigee <b>{row['perigee_km']:.0f} km</b>, "
        f"apogee <b>{row['apogee_km']:.0f} km</b>), this orbit departs strongly from a circular path. "
        "High eccentricity typically concentrates dwell time near apogee or reflects transfer/debris dynamics."
    )
  elif 96 <= row["inc_deg"] <= 102 and row["alt_km"] < 1200:
    pattern_tag = "SURPRISING PATTERN: SUN-SYNCHRONOUS BAND"
    pattern_desc = (
        f"Sitting near <b>{row['inc_deg']:.2f}°</b> inclination and <b>{row['alt_km']:.0f} km</b> altitude, "
        "this orbit leverages Earth's oblateness (J2 nodal precession) so the orbital plane drifts ~0.9856°/day—"
        "matching Earth's motion around the Sun to keep local solar time nearly constant across passes."
    )
  elif 45 <= row["inc_deg"] <= 56 and 250 <= row["alt_km"] <= 1300:
    pattern_tag = "SURPRISING PATTERN: CONSTELLATION / LAUNCH CORRIDOR"
    pattern_desc = (
        f"Clustered around <b>{row['inc_deg']:.2f}°</b> inclination and <b>{row['alt_km']:.0f} km</b> altitude, "
        "this geometry reflects deliberate multi-satellite coverage shells or shared launch inclination constraints."
    )
  else:
    diff = row["alt_km"] - df["alt_km"].mean()
    pattern_tag = "PATTERN INSIGHT: GROUP COMPARISON"
    pattern_desc = (
        f"Across the loaded set, mean inclination is <b>{df['inc_deg'].mean():.2f}°</b> and mean altitude is "
        f"<b>{df['alt_km'].mean():.0f} km</b>. This satellite sits <b>{diff:+.0f} km</b> relative to the group average."
    )

  drag_desc = (
      f"<b>Atmospheric Drag (BSTAR):</b> <code>{row['bstar']:.2e}</code> — "
      "captures upper-atmosphere air-density sensitivity and long-term orbital decay."
  )

  cards_html = f"""
    <div style="width:100%;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">{category_label}</div>
          <div style="font-size:20px;font-weight:700;color:#f8fafc;margin-top:2px;">{row['name']} <span style="font-size:13px;color:#94a3b8;">(NORAD {row['norad_id']})</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:5px 12px;border-radius:999px;font-size:12px;font-weight:700;">{regime}</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">MEAN ALTITUDE</div>
          <div style="font-size:18px;font-weight:700;color:#38bdf8;margin-top:4px;">{row['alt_km']:.1f} km</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">INCLINATION</div>
          <div style="font-size:18px;font-weight:700;color:#a78bfa;margin-top:4px;">{row['inc_deg']:.2f}°</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ECCENTRICITY</div>
          <div style="font-size:18px;font-weight:700;color:#34d399;margin-top:4px;">{row['ecc']:.5f}</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ORBITAL PERIOD</div>
          <div style="font-size:18px;font-weight:700;color:#fbbf24;margin-top:4px;">{row['period_min']:.1f} min</div>
        </div>
      </div>
    </div>
    """

  interp_html = f"""
    <div style="width:100%;font-family:Inter,system-ui,sans-serif;background:#111724;border:1px solid #263248;border-radius:12px;padding:16px 18px;color:#e2e8f0;margin-top:12px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;">
        <span style="font-size:14px;font-weight:700;color:#f8fafc;">🛰️ ORBITAL PATTERN & ASTRODYNAMICS INTERPRETATION</span>
        <span style="background:#1e1b4b;color:#c4b5fd;border:1px solid #6366f1;padding:3px 10px;border-radius:999px;font-size:11px;font-weight:700;">{pattern_tag}</span>
      </div>
      <p style="margin:0 0 10px 0;line-height:1.55;color:#cbd5e1;font-size:14px;">{pattern_desc}</p>
      <p style="margin:0;line-height:1.5;color:#94a3b8;font-size:13px;">{drag_desc}</p>
    </div>
    """
  return cards_html, interp_html


full_width_css = """
body, .gradio-container {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  padding: 12px 18px !important;
}
.gradio-container .main, .gradio-container .app {
  max-width: 100% !important;
  width: 100% !important;
}
"""

initial_cat = "Special-Interest Satellites → 100 (or so) Brightest"
initial_tle = SAMPLE_TLE_BY_CATEGORY[initial_cat]
initial_df = parse_tle(initial_tle)
initial_choices = [
    f"{r['name']} ({r['norad_id']})" for _, r in initial_df.iterrows()
]

with gr.Blocks(css=full_width_css, theme=gr.themes.Soft()) as demo:
  gr.Markdown("## 🛰️ Satellite Orbit & Pattern Analysis Studio")

  with gr.Row():
    with gr.Column(scale=5, min_width=320):
      cat_dd = gr.Dropdown(
          choices=list(SAMPLE_TLE_BY_CATEGORY.keys()),
          value=initial_cat,
          label="CelesTrak Category / Header",
      )
      tle_in = gr.Textbox(
          value=initial_tle,
          lines=8,
          label="TLE Data (Optional Paste — Preloaded with Sample TLEs)",
          placeholder="Paste TLE lines from CelesTrak here...",
      )
      sat_dd = gr.Dropdown(
          choices=initial_choices,
          value=initial_choices[0] if initial_choices else None,
          label="Select Satellite",
      )
      run_btn = gr.Button(
          "⚡ Run Orbit & Pattern Analysis", variant="primary"
      )

    with gr.Column(scale=7, min_width=420):
      metrics_out = gr.HTML()
      commentary_out = gr.HTML()

  cat_dd.change(
      fn=on_category_change, inputs=[cat_dd], outputs=[tle_in, sat_dd]
  )
  tle_in.change(fn=sync_sat_dropdown, inputs=[tle_in], outputs=[sat_dd])
  run_btn.click(
      fn=run_analysis,
      inputs=[cat_dd, tle_in, sat_dd],
      outputs=[metrics_out, commentary_out],
  )

demo.launch(share=False)

/tmp/ipykernel_2830/2979864594.py:242: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=full_width_css, theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [6]:
!pip install -q sgp4 gradio pandas

import math
import gradio as gr
import pandas as pd
from sgp4.api import Satrec

MU_EARTH = 398600.4418
R_EARTH = 6378.137

CATEGORIES = [
    "Communications Satellites → Active Geosynchronous",
    "Communications Satellites → Movers",
    "Communications Satellites → GEO Protected Zone",
    "Communications Satellites → GEO Protected Zone Plus",
    "Communications Satellites → Intelsat",
    "Communications Satellites → SES",
    "Communications Satellites → Eutelsat",
    "Communications Satellites → Telesat",
    "Communications Satellites → Starlink",
    "Communications Satellites → OneWeb",
    "Communications Satellites → Qianfan",
    "Communications Satellites → Hulianwang Digui",
    "Communications Satellites → Kuiper",
    "Communications Satellites → Iridium NEXT",
    "Communications Satellites → Orbcomm",
    "Communications Satellites → Globalstar",
    "Communications Satellites → Amateur Radio",
    "Communications Satellites → SatNOGS",
    "Communications Satellites → Experimental Comm",
    "Communications Satellites → Other Comm",
]


def parse_tle(text):
  lines = [ln.strip() for ln in (text or "").strip().splitlines() if ln.strip()]
  rows = []
  i = 0
  while i + 1 < len(lines):
    if (
        not lines[i].startswith("1 ")
        and lines[i + 1].startswith("1 ")
        and (i + 2) < len(lines)
        and lines[i + 2].startswith("2 ")
    ):
      name, l1, l2 = lines[i], lines[i + 1], lines[i + 2]
      i += 3
    elif lines[i].startswith("1 ") and (i + 1) < len(lines) and lines[i + 1].startswith("2 "):
      l1, l2 = lines[i], lines[i + 1]
      name = f"SAT-{l1[2:7].strip()}"
      i += 2
    else:
      i += 1
      continue
    try:
      sat = Satrec.twoline2rv(l1, l2)
      n_rad_s = sat.no_kozai / 60.0
      if n_rad_s <= 0:
        continue
      inc = math.degrees(sat.inclo)
      ecc = sat.ecco
      period = (2 * math.pi / n_rad_s) / 60.0
      a_km = (MU_EARTH / (n_rad_s**2)) ** (1 / 3)
      per = a_km * (1 - ecc) - R_EARTH
      apo = a_km * (1 + ecc) - R_EARTH
      alt = (per + apo) / 2
      rows.append({
          "name": name,
          "norad_id": sat.satnum,
          "inc_deg": inc,
          "ecc": ecc,
          "period_min": period,
          "perigee_km": per,
          "apogee_km": apo,
          "alt_km": alt,
          "bstar": sat.bstar,
      })
    except Exception:
      continue
  return pd.DataFrame(rows)


def sync_sat_dropdown(tle_text):
  df = parse_tle(tle_text)
  if df.empty:
    return gr.update(choices=[], value=None)
  choices = [f"{r['name']} ({r['norad_id']})" for _, r in df.iterrows()]
  return gr.update(choices=choices, value=choices[0])


def run_analysis(category_label, tle_text, selected_sat):
  df = parse_tle(tle_text)
  if df.empty:
    err_html = """
        <div style="width:100%;box-sizing:border-box;padding:18px;border-radius:12px;background:#161b26;color:#f87171;border:1px solid #334155;font-family:Inter,sans-serif;">
          No TLE data provided or no valid Line 1 / Line 2 entries found. Paste your CelesTrak TLE text first.
        </div>
        """
    return err_html, err_html

  idx = 0
  if selected_sat:
    matches = df.index[
        df.apply(lambda r: f"{r['name']} ({r['norad_id']})" == selected_sat, axis=1)
    ].tolist()
    if matches:
      idx = matches[0]
  row = df.loc[idx]

  if row["alt_km"] < 2000:
    regime = "Low Earth Orbit (LEO)"
  elif row["alt_km"] < 35000:
    regime = "Medium Earth Orbit (MEO)"
  else:
    regime = "Geostationary / High Regime (GEO/HEO)"

  sso_count = int(
      ((df["inc_deg"].between(96, 102)) & (df["alt_km"] < 1200)).sum()
  )
  geo_count = int((df["alt_km"].between(35000, 36600)).sum())
  ecc_count = int((df["ecc"] > 0.05).sum())

  if 35000 <= row["alt_km"] <= 36600 and row["ecc"] < 0.02:
    pattern_tag = "SURPRISING PATTERN: GEOSTATIONARY BELT"
    pattern_desc = (
        f"In your pasted list ({len(df)} satellites), <b>{geo_count}</b> sit in the ~35,786 km geostationary belt "
        f"(period ~1436 min / 1 sidereal day). This satellite (<b>{row['alt_km']:.0f} km</b>, inclination <b>{row['inc_deg']:.2f}°</b>) "
        "matches Earth's rotation rate, appearing nearly stationary over a fixed longitude—ideal for continuous regional communications."
    )
  elif row["ecc"] > 0.05:
    pattern_tag = "SURPRISING PATTERN: ECCENTRIC / TRANSFER ORBIT"
    pattern_desc = (
        f"This satellite has eccentricity <b>{row['ecc']:.4f}</b> (perigee <b>{row['perigee_km']:.0f} km</b>, "
        f"apogee <b>{row['apogee_km']:.0f} km</b>), while <b>{len(df) - ecc_count}</b> of <b>{len(df)}</b> satellites in your list "
        "follow near-circular paths. This points to a geostationary transfer orbit (GTO), graveyard/disposal drift, or high-latitude dwell geometry."
    )
  elif 96 <= row["inc_deg"] <= 102 and row["alt_km"] < 1200:
    pattern_tag = "SURPRISING PATTERN: SUN-SYNCHRONOUS BAND"
    pattern_desc = (
        f"In your pasted list ({len(df)} satellites), <b>{sso_count}</b> fall into the near-polar sun-synchronous band (~96°–102°, LEO). "
        f"This satellite (<b>{row['inc_deg']:.2f}°</b>, <b>{row['alt_km']:.0f} km</b>) uses Earth's J2 oblateness so its orbital plane precesses "
        "~0.9856°/day—common among amateur, CubeSat, and experimental comms payloads."
    )
  elif 45 <= row["inc_deg"] <= 90 and 250 <= row["alt_km"] <= 1600:
    pattern_tag = "SURPRISING PATTERN: LEO CONSTELLATION SHELL"
    pattern_desc = (
        f"Clustered at <b>{row['inc_deg']:.2f}°</b> inclination and <b>{row['alt_km']:.0f} km</b> altitude (group mean: "
        f"<b>{df['inc_deg'].mean():.2f}°</b>, <b>{df['alt_km'].mean():.0f} km</b>), this satellite sits within an organized LEO broadband/constellation shell "
        "designed for low-latency global or mid-latitude coverage."
    )
  else:
    diff = row["alt_km"] - df["alt_km"].mean()
    pattern_tag = "PATTERN INSIGHT: GROUP DISTRIBUTION"
    pattern_desc = (
        f"Across all <b>{len(df)}</b> parsed satellites, mean inclination is <b>{df['inc_deg'].mean():.2f}°</b> and mean altitude is "
        f"<b>{df['alt_km'].mean():.0f} km</b>. This satellite sits <b>{diff:+.0f} km</b> relative to the group's mean altitude."
    )

  drag_desc = (
      f"<b>Parsed from Line 1 & Line 2:</b> Perigee <code>{row['perigee_km']:.1f} km</code>, "
      f"Apogee <code>{row['apogee_km']:.1f} km</code>, Atmospheric Drag (<code>BSTAR</code>) <code>{row['bstar']:.2e}</code>."
  )

  cards_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">{category_label} • {len(df)} SATELLITES PARSED</div>
          <div style="font-size:21px;font-weight:700;color:#f8fafc;margin-top:2px;">{row['name']} <span style="font-size:13px;color:#94a3b8;">(NORAD {row['norad_id']})</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:6px 12px;border-radius:999px;font-size:12px;font-weight:700;">{regime}</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">MEAN ALTITUDE</div>
          <div style="font-size:19px;font-weight:700;color:#38bdf8;margin-top:4px;">{row['alt_km']:.1f} km</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">INCLINATION</div>
          <div style="font-size:19px;font-weight:700;color:#a78bfa;margin-top:4px;">{row['inc_deg']:.2f}°</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ECCENTRICITY</div>
          <div style="font-size:19px;font-weight:700;color:#34d399;margin-top:4px;">{row['ecc']:.5f}</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">ORBITAL PERIOD</div>
          <div style="font-size:19px;font-weight:700;color:#fbbf24;margin-top:4px;">{row['period_min']:.1f} min</div>
        </div>
      </div>
    </div>
    """

  interp_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;background:#111724;border:1px solid #263248;border-radius:12px;padding:16px 18px;color:#e2e8f0;margin-top:12px;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;margin-bottom:10px;">
        <span style="font-size:14px;font-weight:700;color:#f8fafc;">🛰️ ORBITAL PATTERN & ASTRODYNAMICS INTERPRETATION</span>
        <span style="background:#1e1b4b;color:#c4b5fd;border:1px solid #6366f1;padding:4px 10px;border-radius:999px;font-size:11px;font-weight:700;">{pattern_tag}</span>
      </div>
      <p style="margin:0 0 10px 0;line-height:1.6;color:#cbd5e1;font-size:14px;">{pattern_desc}</p>
      <p style="margin:0;line-height:1.5;color:#94a3b8;font-size:13px;">{drag_desc}</p>
    </div>
    """
  return cards_html, interp_html


full_width_css = """
html, body, .gradio-container, .gradio-container > .main, .gradio-container > .main > .wrap, .block, .form {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}
.gradio-container {
  padding: 10px 14px !important;
}
"""

with gr.Blocks() as demo:
  gr.Markdown("## 🛰️ Communications Satellites — Orbit & Pattern Studio")

  with gr.Row(equal_height=False):
    with gr.Column(scale=5, min_width=320):
      cat_dd = gr.Dropdown(
          choices=CATEGORIES,
          value=CATEGORIES[0],
          label="Communications Satellites Category",
      )
      tle_in = gr.Textbox(
          value="",
          lines=12,
          label="Paste TLE Data",
          placeholder="Paste your CelesTrak TLE data here...",
      )
      sat_dd = gr.Dropdown(
          choices=[], value=None, label="Select Satellite from Pasted Data"
      )
      run_btn = gr.Button(
          "⚡ Run Orbit & Pattern Analysis", variant="primary"
      )

    with gr.Column(scale=7, min_width=420):
      metrics_out = gr.HTML()
      commentary_out = gr.HTML()

  tle_in.change(fn=sync_sat_dropdown, inputs=[tle_in], outputs=[sat_dd])
  run_btn.click(
      fn=run_analysis,
      inputs=[cat_dd, tle_in, sat_dd],
      outputs=[metrics_out, commentary_out],
  )

demo.launch(share=False, theme=gr.themes.Soft(), css=full_width_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>